In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures

In [2]:
df = pd.read_csv('data/merged/train_processed.csv')
df.describe()

,pickup_longitude_rounded,pickup_latitude_rounded,dropoff_longitude_rounded,dropoff_latitude_rounded,passenger_count,trip_duration,fare_amount
count,126473.000000,126473.000000,126473.000000,126473.000000,126473.000000,1.264730e+05,126473.000000
mean,-73.976081,40.753639,-73.977687,40.754277,1.183850,7.674804e+02,9.233110
std,0.031296,0.022355,0.027004,0.022086,0.671001,5.578705e+03,8.339225
min,-74.194000,40.616000,-74.194000,40.602000,1.000000,1.000000e+00,-5.500000
25%,-73.991000,40.742000,-73.991000,40.742000,1.000000,3.310000e+02,5.300000
50%,-73.982000,40.756000,-73.981000,40.756000,1.000000,5.240000e+02,6.900000
75%,-73.969000,40.768000,-73.970000,40.768000,1.000000,8.280000e+02,9.500000
max,-73.422000,41.022000,-73.422000,41.022000,6.000000,1.763934e+06,152.450000


# Fare Price

In [25]:
# Linear Regression
X = df.drop(columns=['trip_duration', 'fare_amount'])
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=151)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Train MSE:', mean_squared_error(y_train, model.predict(X_train)))
print('Test MSE:', mean_squared_error(y_test, model.predict(X_test)))

Train MSE: 14.74938073659351
Test MSE: 14.23588389923588


In [23]:
# Polynomial Regression
X = df.drop(columns=['trip_duration', 'fare_amount'])
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=151)

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.fit_transform(X_test)

model = LinearRegression()
model.fit(X_train_poly, y_train)

print('Train MSE:', mean_squared_error(y_train, model.predict(X_train_poly)))
print('Test MSE:', mean_squared_error(y_test, model.predict(X_test_poly)))

Train MSE: 10.949308323201521
Test MSE: 11.703013793201306


In [24]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, model.predict(X_test_poly))
print(f'R-squared: {r2}')

R-squared: 0.8349898492253713


# Duration

In [4]:
df['manhattan_distance'] = np.abs(df['pickup_latitude_rounded'] - df['dropoff_latitude_rounded']) + np.abs(df['pickup_longitude_rounded'] - df['dropoff_longitude_rounded'])

In [10]:
# Linear Regression
# X = df.drop(columns=['trip_duration', 'fare_amount'])
X = df[['manhattan_distance']]
y = df['trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Train MSE:', mean_squared_error(y_train, model.predict(X_train)))
print('Test MSE:', mean_squared_error(y_test, model.predict(X_test)))

Train MSE: 5966458.376525451
Test MSE: 130464950.02235694


In [19]:
df.head()

,pickup_longitude_rounded,pickup_latitude_rounded,dropoff_longitude_rounded,dropoff_latitude_rounded,passenger_count,trip_duration,fare_amount,manhattan_distance
0,-74.194,40.684,-74.194,40.684,1,33.0,52.00,0.0
1,-74.183,40.688,-74.183,40.688,1,69.0,79.75,0.0
2,-74.183,40.688,-74.183,40.688,2,11.0,86.50,0.0
3,-74.183,41.022,-74.183,41.022,1,73.0,152.45,0.0
4,-74.182,40.688,-74.182,40.688,1,3.5,80.00,0.0


In [20]:
# Polynomial Regression
# X = df.drop(columns=['trip_duration', 'fare_amount'])
data = df[['manhattan_distance', 'trip_duration']]
data = data[(data['trip_duration'] <= 1000) & (data['manhattan_distance'] <= 0.05)]

X = data[['manhattan_distance']]
y = data['trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.fit_transform(X_test)

model = LinearRegression()
model.fit(X_train_poly, y_train)

print('Train MSE:', mean_squared_error(y_train, model.predict(X_train_poly)))
print('Test MSE:', mean_squared_error(y_test, model.predict(X_test_poly)))

Train MSE: 32953.64674637958
Test MSE: 32813.26620769734


In [21]:
r2 = r2_score(y_test, model.predict(X_test_poly))
print(f'R-squared: {r2}')

R-squared: 0.34250330790224714


In [8]:
y_pred = model.predict(X_test)

print(pd.DataFrame(y_pred - y_test).describe())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


ValueError: X has 6 features, but LinearRegression is expecting 462 features as input.